# Standalone LSTM Notebook

This notebook is fully standalone and does not import from the repo. It only needs the SQLite dataset file.

Estimated training time on the current dataset:
- Colab GPU, full dataset, 5 CV folds + final fit: about 45-110 minutes
- CPU only: about 3-6 hours
- `max_tickers=200`: often 10-25 minutes

The saved `.pkl` payload is still compatible with the pipeline's `lstm` loader.


In [ ]:
# Runtime -> Change runtime type -> Hardware accelerator -> GPU
!pip -q install pandas numpy scikit-learn torch joblib


In [ ]:
from pathlib import Path

# Point this to your SQLite file in Colab or Drive.
DB_PATH = Path('/content/stock_prices_20y.db')

# Standalone outputs. These artifacts are still pipeline-compatible.
OUTPUT_DIR = Path('/content/model_outputs')
MODELS_DIR = OUTPUT_DIR / 'models'
RESULTS_DIR = OUTPUT_DIR / 'results'
CONFIGS_DIR = OUTPUT_DIR / 'configs'

for path in [MODELS_DIR, RESULTS_DIR, CONFIGS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(f'Missing SQLite database: {DB_PATH}')

print('DB_PATH    =', DB_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
import os
import sqlite3
import time
from datetime import datetime

import numpy as np
import pandas as pd
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_ENABLED = DEVICE.type == 'cuda'
print('Torch device:', DEVICE)
if GPU_ENABLED:
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
else:
    print('No GPU detected. The notebook still runs, just slower.')

with sqlite3.connect(DB_PATH) as conn:
    row_count = conn.execute('SELECT COUNT(*) FROM prices').fetchone()[0]
    ticker_count = conn.execute('SELECT COUNT(DISTINCT ticker) FROM prices').fetchone()[0]
    min_date, max_date = conn.execute('SELECT MIN(date), MAX(date) FROM prices').fetchone()

print(f'Rows: {row_count:,} | Tickers: {ticker_count:,} | Date range: {min_date} -> {max_date}')


In [ ]:
import gc
import json
import pickle
import random
import sqlite3
from dataclasses import asdict, dataclass
from typing import List, Tuple

import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

NUM_WORKERS = min(2, os.cpu_count() or 0)
AMP_ENABLED = DEVICE.type == 'cuda'
RUN_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S')


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_prices_from_sqlite(db_path: str, table_name: str) -> pd.DataFrame:
    conn = sqlite3.connect(db_path)
    query = f'''        SELECT ticker, date, open, high, low, close, adj_close, volume
        FROM {table_name}
        ORDER BY ticker, date
    '''
    df = pd.read_sql_query(query, conn)
    conn.close()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
    float_cols = ['open', 'high', 'low', 'close', 'adj_close']
    for c in float_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')
    df['volume'] = pd.to_numeric(df['volume'], errors='coerce').fillna(0).astype('float32')
    return df


def split_train_test_per_ticker(df: pd.DataFrame, test_size: float, embargo: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_parts, test_parts = [], []
    for _, grp in df.groupby('ticker', sort=False):
        grp = grp.sort_values('date').reset_index(drop=True)
        n = len(grp)
        split_idx = int(n * (1 - test_size))
        train_end = split_idx - embargo
        if split_idx <= 0 or split_idx >= n or train_end <= 0:
            continue
        train_parts.append(grp.iloc[:train_end].copy())
        test_parts.append(grp.iloc[split_idx:].copy())
    if not train_parts or not test_parts:
        raise ValueError('Empty train/test split.')
    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)


def build_grouped_time_folds(meta: pd.DataFrame, n_splits: int, embargo: int) -> List[Tuple[np.ndarray, np.ndarray]]:
    meta = meta.sort_values(['ticker', 'date']).reset_index(drop=True)
    folds = []
    for fold_num in range(n_splits):
        tr_idx_all, va_idx_all = [], []
        for _, grp in meta.groupby('ticker', sort=False):
            idx = grp.index.to_numpy()
            n = len(idx)
            if n < (n_splits + 1) * max(embargo, 1) + 5:
                continue
            boundaries = np.linspace(0, n, n_splits + 2, dtype=int)
            va_start = boundaries[fold_num + 1]
            va_end = boundaries[fold_num + 2]
            purged_train_end = va_start - embargo
            if purged_train_end <= 0 or va_end <= va_start:
                continue
            tr_idx_all.extend(idx[:purged_train_end].tolist())
            va_idx_all.extend(idx[va_start:va_end].tolist())
        if tr_idx_all and va_idx_all:
            folds.append((np.asarray(tr_idx_all, dtype=np.int64), np.asarray(va_idx_all, dtype=np.int64)))
    if not folds:
        raise ValueError('No CV folds built.')
    return folds


def fit_scaler(X_all: np.ndarray, indices: np.ndarray, chunk_rows: int = 50000) -> StandardScaler:
    scaler = StandardScaler()
    n_features = X_all.shape[2]
    for start in range(0, len(indices), chunk_rows):
        batch_idx = indices[start:start + chunk_rows]
        x_chunk = X_all[batch_idx].reshape(-1, n_features)
        scaler.partial_fit(x_chunk)
    return scaler


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) < 2:
        return {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan, 'directional_accuracy': np.nan}
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'directional_accuracy': float(np.mean(np.sign(y_true) == np.sign(y_pred))),
    }


class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, indices: np.ndarray, scaler: StandardScaler):
        self.X = X
        self.y = y
        self.indices = indices
        self.mean = scaler.mean_.astype(np.float32)
        self.scale = scaler.scale_.astype(np.float32)
        self.scale[self.scale == 0.0] = 1.0

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        x = self.X[i].copy()
        x = (x - self.mean) / self.scale
        return torch.from_numpy(x), torch.tensor(self.y[i], dtype=torch.float32)


def make_loader(dataset, batch_size, shuffle):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=AMP_ENABLED,
        persistent_workers=NUM_WORKERS > 0,
    )


def train_model_amp(model, train_loader, val_loader, cfg):
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.learning_rate * 0.05)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        total_loss, n_train = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=AMP_ENABLED)
            yb = yb.to(DEVICE, non_blocking=AMP_ENABLED)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                pred = model(xb)
                loss = criterion(pred, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * len(yb)
            n_train += len(yb)
        scheduler.step()
        train_loss = total_loss / max(n_train, 1)
        model.eval()
        total_val, n_val = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE, non_blocking=AMP_ENABLED)
                yb = yb.to(DEVICE, non_blocking=AMP_ENABLED)
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                    pred = model(xb)
                    val_loss = criterion(pred, yb)
                total_val += val_loss.item() * len(yb)
                n_val += len(yb)
        val_loss = total_val / max(n_val, 1)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if epoch % 5 == 0 or patience_counter == 0:
            print(f'Epoch {epoch:3d} | train={train_loss:.6f} | val={val_loss:.6f} | lr={scheduler.get_last_lr()[0]:.2e}', flush=True)
        if patience_counter >= cfg.patience:
            print(f'Early stop at epoch {epoch}')
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    preds = []
    for xb, _ in loader:
        xb = xb.to(DEVICE, non_blocking=AMP_ENABLED)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
            preds.append(model(xb).detach().cpu().numpy())
    return np.concatenate(preds)

@dataclass
class Config:
    db_path: str = str(DB_PATH)
    table_name: str = 'prices'
    n_splits: int = 5
    test_size: float = 0.2
    target_horizon: int = 1
    predict_target: str = 'return'
    random_state: int = 42
    sequence_length: int = 30
    batch_size: int = 4096 if GPU_ENABLED else 2048
    epochs: int = 60
    patience: int = 10
    lstm_units: int = 128
    num_layers: int = 2
    dense_units: int = 64
    dropout: float = 0.3
    learning_rate: float = 5e-4
    weight_decay: float = 1e-4
    max_tickers: int = 0
    model_output_path: str = str(MODELS_DIR / f'{RUN_TAG}-rnn_torch.pkl')
    config_output_path: str = str(CONFIGS_DIR / f'{RUN_TAG}-rnn_torch_config.json')
    feature_cols: tuple = (
        'log_ret_1', 'log_ret_3', 'log_ret_5', 'log_ret_10', 'log_ret_20',
        'hl_range', 'oc_change', 'close_vs_ma_5', 'close_vs_ma_10',
        'close_vs_ma_20', 'close_vs_ma_50', 'ma_5_vs_ma_20',
        'rolling_vol_5', 'rolling_vol_20', 'log_vol_chg_1',
        'vol_vs_ma_5', 'vol_vs_ma_20', 'rsi_14', 'macd_norm',
        'bb_pos', 'atr_14_norm',
    )

cfg = Config()
print(cfg)


def make_sequence_features(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for _, g in df.groupby('ticker', sort=False):
        g = g.sort_values('date').copy()
        close = g['close'].astype('float64')
        high = g['high'].astype('float64')
        low = g['low'].astype('float64')
        open_ = g['open'].astype('float64')
        volume = g['volume'].astype('float64').clip(lower=1.0)
        log_close = np.log(close)
        g['log_ret_1'] = log_close.diff(1)
        g['log_ret_3'] = log_close.diff(3)
        g['log_ret_5'] = log_close.diff(5)
        g['log_ret_10'] = log_close.diff(10)
        g['log_ret_20'] = log_close.diff(20)
        g['hl_range'] = (high - low) / close
        g['oc_change'] = (close - open_) / open_.clip(lower=1e-6)
        ma5 = close.rolling(5).mean()
        ma10 = close.rolling(10).mean()
        ma20 = close.rolling(20).mean()
        ma50 = close.rolling(50).mean()
        g['close_vs_ma_5'] = close / ma5.clip(lower=1e-6) - 1.0
        g['close_vs_ma_10'] = close / ma10.clip(lower=1e-6) - 1.0
        g['close_vs_ma_20'] = close / ma20.clip(lower=1e-6) - 1.0
        g['close_vs_ma_50'] = close / ma50.clip(lower=1e-6) - 1.0
        g['ma_5_vs_ma_20'] = ma5 / ma20.clip(lower=1e-6) - 1.0
        g['rolling_vol_5'] = g['log_ret_1'].rolling(5).std()
        g['rolling_vol_20'] = g['log_ret_1'].rolling(20).std()
        log_vol = np.log1p(volume)
        g['log_vol_chg_1'] = log_vol.diff(1)
        vol_ma5 = volume.rolling(5).mean()
        vol_ma20 = volume.rolling(20).mean()
        g['vol_vs_ma_5'] = volume / vol_ma5.clip(lower=1.0) - 1.0
        g['vol_vs_ma_20'] = volume / vol_ma20.clip(lower=1.0) - 1.0
        delta = close.diff()
        gain = delta.clip(lower=0).ewm(alpha=1.0 / 14, min_periods=14).mean()
        loss = (-delta.clip(upper=0)).ewm(alpha=1.0 / 14, min_periods=14).mean()
        g['rsi_14'] = (100.0 - 100.0 / (1.0 + gain / (loss + 1e-8))) / 100.0 - 0.5
        ema12 = close.ewm(span=12, min_periods=12).mean()
        ema26 = close.ewm(span=26, min_periods=26).mean()
        g['macd_norm'] = (ema12 - ema26) / close.clip(lower=1e-6)
        bb_mean = close.rolling(20).mean()
        bb_std = close.rolling(20).std()
        bb_upper = bb_mean + 2 * bb_std
        bb_lower = bb_mean - 2 * bb_std
        g['bb_pos'] = (close - bb_lower) / (bb_upper - bb_lower).clip(lower=1e-6) - 0.5
        prev_close = close.shift(1)
        tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
        g['atr_14_norm'] = tr.ewm(alpha=1.0 / 14, min_periods=14).mean() / close.clip(lower=1e-6)
        out.append(g)
    feat_df = pd.concat(out, ignore_index=True)
    return feat_df.replace([np.inf, -np.inf], np.nan)


def add_target(df: pd.DataFrame, horizon: int, predict_target: str) -> pd.DataFrame:
    out = []
    for _, g in df.groupby('ticker', sort=False):
        g = g.sort_values('date').copy()
        close = g['close']
        if predict_target == 'return':
            g['target'] = np.log(close.shift(-horizon) / close).astype('float32')
        elif predict_target == 'price':
            g['target'] = close.shift(-horizon).astype('float32')
        else:
            raise ValueError("predict_target must be 'return' or 'price'")
        out.append(g)
    return pd.concat(out, ignore_index=True)


def build_sequences(df: pd.DataFrame, feature_cols: List[str], sequence_length: int) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    X_list, y_list, meta_list = [], [], []
    for ticker, g in df.groupby('ticker', sort=False):
        g = g.sort_values('date').reset_index(drop=True)
        if len(g) < sequence_length:
            continue
        feat = g[feature_cols].to_numpy(dtype=np.float32)
        target = g['target'].to_numpy(dtype=np.float32)
        dates = g['date'].to_numpy()
        closes = g['close'].to_numpy(dtype=np.float32)
        for i in range(sequence_length - 1, len(g)):
            if np.isnan(target[i]):
                continue
            x_seq = feat[i - sequence_length + 1:i + 1]
            if np.isnan(x_seq).any():
                continue
            X_list.append(x_seq)
            y_list.append(target[i])
            meta_list.append((ticker, dates[i], closes[i]))
    if not X_list:
        raise ValueError('No sequences created.')
    return np.asarray(X_list, dtype=np.float32), np.asarray(y_list, dtype=np.float32), pd.DataFrame(meta_list, columns=['ticker', 'date', 'close'])


class LSTMModel(nn.Module):
    def __init__(self, n_features: int, cfg: Config):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=cfg.lstm_units, num_layers=cfg.num_layers, batch_first=True, nonlinearity='tanh', dropout=cfg.dropout if cfg.num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(cfg.dropout)
        self.fc1 = nn.Linear(cfg.lstm_units, cfg.dense_units)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(cfg.dense_units, cfg.dense_units // 2)
        self.fc3 = nn.Linear(cfg.dense_units // 2, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last = lstm_out[:, -1, :]
        out = self.dropout(last)
        out = self.act(self.fc1(out))
        out = self.act(self.fc2(out))
        return self.fc3(out).squeeze(-1)


In [ ]:
start_time = time.perf_counter()
set_seed(cfg.random_state)

print('Loading data...')
raw_df = load_prices_from_sqlite(cfg.db_path, cfg.table_name)
all_tickers = list(raw_df['ticker'].unique())
if cfg.max_tickers > 0 and len(all_tickers) > cfg.max_tickers:
    rng = np.random.default_rng(cfg.random_state)
    all_tickers = rng.choice(all_tickers, cfg.max_tickers, replace=False).tolist()
    raw_df = raw_df[raw_df['ticker'].isin(all_tickers)].reset_index(drop=True)
    print(f'Subsampled to {cfg.max_tickers} tickers')
else:
    print(f'Using all {len(all_tickers)} tickers')

feat_df = make_sequence_features(raw_df)
train_feat_df, test_feat_df = split_train_test_per_ticker(feat_df, cfg.test_size, cfg.target_horizon)
train_df = add_target(train_feat_df, cfg.target_horizon, cfg.predict_target)
test_df = add_target(test_feat_df, cfg.target_horizon, cfg.predict_target)
X_train_raw, y_train, meta_train = build_sequences(train_df, list(cfg.feature_cols), cfg.sequence_length)
X_test_raw, y_test, meta_test = build_sequences(test_df, list(cfg.feature_cols), cfg.sequence_length)
print(f'Train sequences: {len(X_train_raw):,} | Test sequences: {len(X_test_raw):,}')

folds = build_grouped_time_folds(meta_train, cfg.n_splits, cfg.target_horizon)
print('CV folds:', len(folds))

n_features = len(cfg.feature_cols)
cv_results = []
for fold_i, (tr_idx, va_idx) in enumerate(folds, start=1):
    print(f'\n=== Fold {fold_i}/{len(folds)} ===')
    scaler = fit_scaler(X_train_raw, tr_idx)
    train_ds = SequenceDataset(X_train_raw, y_train, tr_idx, scaler)
    val_ds = SequenceDataset(X_train_raw, y_train, va_idx, scaler)
    train_loader = make_loader(train_ds, cfg.batch_size, True)
    val_loader = make_loader(val_ds, cfg.batch_size, False)
    model = LSTMModel(n_features, cfg).to(DEVICE)
    model = train_model_amp(model, train_loader, val_loader, cfg)
    metrics = regression_metrics(y_train[va_idx], predict_model(model, val_loader))
    cv_results.append(metrics)
    print(metrics)
    del model, scaler, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    if GPU_ENABLED:
        torch.cuda.empty_cache()

cv_df = pd.DataFrame(cv_results)
print('\nCV mean metrics')
print(cv_df.mean(numeric_only=True).to_string())

all_idx = np.arange(len(X_train_raw), dtype=np.int64)
scaler = fit_scaler(X_train_raw, all_idx)
split = int(len(all_idx) * 0.9)
train_ds = SequenceDataset(X_train_raw, y_train, all_idx[:split], scaler)
val_ds = SequenceDataset(X_train_raw, y_train, all_idx[split:], scaler)
train_loader = make_loader(train_ds, cfg.batch_size, True)
val_loader = make_loader(val_ds, cfg.batch_size, False)
final_model = LSTMModel(n_features, cfg).to(DEVICE)
final_model = train_model_amp(final_model, train_loader, val_loader, cfg)

test_idx = np.arange(len(X_test_raw), dtype=np.int64)
test_ds = SequenceDataset(X_test_raw, y_test, test_idx, scaler)
test_loader = make_loader(test_ds, cfg.batch_size, False)
y_test_pred = predict_model(final_model, test_loader)
test_metrics = regression_metrics(y_test, y_test_pred)
print('Held-out test metrics:', test_metrics)

cpu_state_dict = {k: v.detach().cpu() for k, v in final_model.state_dict().items()}
payload = {'model_state_dict': cpu_state_dict, 'scaler': scaler, 'config': asdict(cfg), 'cv_results': cv_results, 'test_metrics': test_metrics}
with open(cfg.model_output_path, 'wb') as f:
    pickle.dump(payload, f)
config_dict = asdict(cfg)
config_dict['feature_cols'] = list(config_dict['feature_cols'])
with open(cfg.config_output_path, 'w') as f:
    json.dump(config_dict, f, indent=2)

pred_path = RESULTS_DIR / f'{RUN_TAG}-lstm_torch_predictions.csv'
out = meta_test.copy()
out['y_true'] = y_test
out['y_pred'] = y_test_pred
out.to_csv(pred_path, index=False)

elapsed_min = (time.perf_counter() - start_time) / 60
print(f'\nSaved model      : {cfg.model_output_path}')
print(f'Saved config     : {cfg.config_output_path}')
print(f'Saved predictions: {pred_path}')
print(f'Elapsed minutes  : {elapsed_min:.2f}')
